# A2 — Pierce knowledge-base demo
This notebook is the reproducible evidence runner for OCR quality and one real retrieval. It refuses to invent labels, a query, or an index result.
Expected label schema: one JSON object per line with `page_id` and hand-corrected `text`; an optional `query` field may supply the real retrieval question.

In [ ]:
from collections import Counter
from pathlib import Path
import json

from doc_agent import config
from doc_agent.contracts import Chunk
from doc_agent.ingest import enhance, loader, preprocess
from doc_agent.index import chunk as chunk_stage
from doc_agent.index import embed, store
from doc_agent.vision import layout, ocr

cfg = config.load()
labels_path = Path('grading_kit/labels.jsonl')
if not labels_path.is_file():
    raise FileNotFoundError(labels_path)
labels = []
for line in labels_path.read_text(encoding='utf-8').splitlines():
    if not line.strip() or line.lstrip().startswith('#'):
        continue
    if 'REPLACE ME' in line:
        raise RuntimeError('Replace the starter label with a real hand-corrected transcription')
    row = json.loads(line)
    if not isinstance(row, dict) or not isinstance(row.get('page_id'), str) or not isinstance(row.get('text'), str):
        raise ValueError('each label must contain page_id and hand-corrected text')
    labels.append(row)
if not labels:
    raise RuntimeError('Add real hand-corrected labels before running A2 evidence')
label_by_page = {row['page_id']: row for row in labels}
print({'label_pages': sorted(label_by_page), 'label_count': len(label_by_page)})


In [ ]:
pages = loader.load_pages(cfg)
pages = preprocess.run(pages, cfg)
pages = enhance.run(pages, cfg)
cfg['pages'] = pages
regions = layout.detect(pages, cfg)
raw_chunks = ocr.transcribe(regions, cfg)
ocr_chunks = chunk_stage.split(raw_chunks, cfg)

def combine_page_text(chunks):
    out = {}
    for chunk in chunks:
        out.setdefault(chunk.page_ids[0], []).append(chunk.text)
    return {page_id: ' '.join(parts) for page_id, parts in out.items()}

predicted = combine_page_text(ocr_chunks)
def word_f1(prediction, gold):
    p, g = Counter(prediction.lower().split()), Counter(gold.lower().split())
    overlap = sum((p & g).values())
    precision = overlap / max(sum(p.values()), 1)
    recall = overlap / max(sum(g.values()), 1)
    return 2 * precision * recall / max(precision + recall, 1e-12)

scores = {page_id: word_f1(predicted.get(page_id, ''), row['text']) for page_id, row in label_by_page.items()}
print({'ocr_word_f1_by_page': scores, 'ocr_word_f1_mean': sum(scores.values()) / max(len(scores), 1)})


In [ ]:
vectors = embed.encode(ocr_chunks, cfg)
store.build(ocr_chunks, vectors, cfg)
index, indexed_chunks, metadata = store.load(cfg)
print({'index': metadata, 'corpus_chunks': len(indexed_chunks), 'embedding_dimension': index.d})

QUERY = next((row.get('query') for row in labels if row.get('query')), None)
if not isinstance(QUERY, str) or not QUERY.strip():
    raise RuntimeError('Set a real question in one label row under query before claiming retrieval evidence')
query_chunk = Chunk(id='query', doc_id=cfg['ingest']['doc_id'], text=QUERY, page_ids=[])
query_vector = embed.encode([query_chunk], cfg)
distances, positions = index.search(query_vector, 1)
position = int(positions[0][0])
if position < 0:
    raise RuntimeError('the configured index returned no candidate')
print({'query': QUERY, 'distance': float(distances[0][0]), 'top_chunk': indexed_chunks[position].model_dump()})


Only the printed scores and retrieved chunk from a run with real labels and a real query belong in the A2 form. Model confidence, detector agreement, and an empty or placeholder label file are not evidence. The current fixed `Chunk` contract retains page provenance but not per-word boxes, so this demo must not claim the explainability target until that contract limitation is resolved.